# Forest Cover Type Classification Using Machine Learning
**CA3 End-to-End Predictive Analytics | KDD Process**  
**Student:** Soumya Dixit | **PRN:** 23070521151

## Project objective
Predict one of seven forest cover types from cartographic and environmental features using supervised multiclass classification.

## Dataset
The notebook uses Scikit-learn's `fetch_covtype`, which loads the UCI Covertype dataset. Dataset reference: [UCI Covertype](https://archive.ics.uci.edu/dataset/31/covertype).

**Computational note:** The original dataset is large. By default, this notebook takes a reproducible stratified sample before modelling so that it can run on a typical student laptop. Report results as results on this sample, not as full-dataset results. Set `SAMPLE_SIZE = None` to use the complete dataset if your computer has enough memory and processing time.

## 1. Install/import libraries

In [ ]:
# If a package is missing, run this in a notebook cell:
# %pip install -U scikit-learn pandas numpy matplotlib seaborn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme()  # Uses the default Seaborn theme for clear plots
pd.set_option("display.max_columns", 60)

## 2. Load the dataset

In [ ]:
# Downloads the dataset the first time; a working internet connection may be required.
covtype = fetch_covtype(as_frame=True)

X_all = covtype.data
y_all = covtype.target

print("Full dataset shape:", X_all.shape)
print("Target shape:", y_all.shape)
print("Number of features:", X_all.shape[1])
print("Target classes:", sorted(y_all.unique()))
display(X_all.head())

## 3. Data understanding

In [ ]:
print("Feature data types:")
display(X_all.dtypes.value_counts().rename_axis("Data type").to_frame("Column count"))

print("Missing values in features:", int(X_all.isna().sum().sum()))
print("Missing target values:", int(y_all.isna().sum()))
print("Duplicate rows:", int(X_all.duplicated().sum()))

print("\nTarget class counts:")
class_counts = y_all.value_counts().sort_index()
display(class_counts.to_frame("Count"))

print("\nTarget class percentages:")
display((100 * y_all.value_counts(normalize=True).sort_index()).round(2).to_frame("Percent"))

## 4. Exploratory data analysis (EDA)

In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values)
plt.title("Distribution of Forest Cover Types")
plt.xlabel("Cover type")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

In [ ]:
# Select a few meaningful numerical features for visual inspection.
continuous_candidates = [
    "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
    "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]
continuous_features = [c for c in continuous_candidates if c in X_all.columns]

eda_sample = X_all[continuous_features].copy()
eda_sample["Cover_Type"] = y_all.astype(str).values
plot_sample = eda_sample.sample(n=min(10000, len(eda_sample)), random_state=RANDOM_STATE)

plt.figure(figsize=(10, 5))
sns.boxplot(data=plot_sample, x="Cover_Type", y="Elevation")
plt.title("Elevation by Forest Cover Type (EDA sample)")
plt.xlabel("Cover type")
plt.ylabel("Elevation")
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [c for c in continuous_features if c in X_all.columns]
plt.figure(figsize=(11, 8))
sns.heatmap(X_all[corr_cols].sample(n=min(20000, len(X_all)), random_state=RANDOM_STATE).corr(),
            cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Selected Numerical Features")
plt.tight_layout()
plt.show()

### EDA interpretation
Use the plots above to describe observed differences between cover types and relationships among numerical variables. Do not claim that a feature causes a cover type; these plots show associations in this dataset.

## 5. Data preparation and sampling

In [ ]:
# Reproducible stratified sample for practical runtime.
# Change to None to use the full dataset if your computer can handle it.
SAMPLE_SIZE = 80000

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(X_all):
    # Stratified sampling preserves approximate proportions of all seven classes.
    X, _, y, _ = train_test_split(
        X_all, y_all,
        train_size=SAMPLE_SIZE,
        stratify=y_all,
        random_state=RANDOM_STATE
    )
else:
    X, y = X_all.copy(), y_all.copy()

print("Rows used for modelling:", len(X))
print("Feature count:", X.shape[1])
print("Class counts in modelling data:")
display(y.value_counts().sort_index().to_frame("Count"))

# This dataset is already numeric; soil and wilderness variables are encoded as binary indicators.
# We use a stratified hold-out test split so every class is represented in both partitions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Model development

The models are compared against a majority-class baseline. Decision Tree and Random Forest use the encoded features directly. KNN uses a pipeline with standardization because distance-based models are sensitive to feature scales. The scaler is fitted on training data only.

In [ ]:
models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=20, min_samples_leaf=2, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=120, min_samples_leaf=1,
        max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "KNN (k=5)": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5, weights="distance", n_jobs=-1))
    ])
}

results = []
fitted_models = {}
predictions = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (macro)", ascending=False)
display(results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision (macro)": "{:.4f}",
    "Recall (macro)": "{:.4f}",
    "F1-score (macro)": "{:.4f}"
}))

## 7. Compare model performance

In [ ]:
metrics_to_plot = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"]
results_long = results_df.melt(id_vars="Model", value_vars=metrics_to_plot,
                               var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 5))
sns.barplot(data=results_long, x="Model", y="Score", hue="Metric")
plt.ylim(0, 1)
plt.title("Model Performance Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8. Classification report for the best model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_predictions = predictions[best_model_name]

print("Best model by macro F1-score:", best_model_name)
print(classification_report(y_test, best_predictions, digits=4, zero_division=0))

## 9. Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions,
    display_labels=sorted(pd.Series(y).unique()),
    cmap="Blues",
    xticks_rotation="vertical",
    ax=ax,
    colorbar=False
)
ax.set_title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 10. Feature importance

In [ ]:
# Tree-based feature importance can help explain which features the model used.
tree_model_name = "Random Forest"
rf_model = fitted_models[tree_model_name]

if hasattr(rf_model, "feature_importances_"):
    importance = pd.Series(rf_model.feature_importances_, index=X_train.columns)
    importance = importance.sort_values(ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(x=importance.values, y=importance.index)
    plt.title("Top 15 Feature Importances — Random Forest")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
    display(importance.to_frame("Importance"))
else:
    print("Feature importance is not available for this model.")

## 11. Results interpretation

In [ ]:
# Automatically create a short factual summary from the measured test metrics.
best_row = results_df.iloc[0]
baseline_row = results_df[results_df["Model"] == "Dummy baseline"].iloc[0]

print(f"Best model by macro F1-score: {best_row['Model']}")
print(f"Best model accuracy: {best_row['Accuracy']:.4f}")
print(f"Best model macro F1-score: {best_row['F1-score (macro)']:.4f}")
print(f"Dummy baseline accuracy: {baseline_row['Accuracy']:.4f}")
print(f"Dummy baseline macro F1-score: {baseline_row['F1-score (macro)']:.4f}")

print("\nInterpretation prompts:")
print("- Compare the best model with the baseline.")
print("- Inspect the classification report to see which cover types are harder to classify.")
print("- Use the confusion matrix to identify commonly confused classes.")
print("- Feature importance indicates predictive contribution, not causation.")

## 12. KDD workflow summary

1. **Data selection:** UCI Covertype dataset.
2. **Data understanding:** Inspected columns, data types, missing values, duplicates, and target distribution.
3. **Preprocessing:** Created a reproducible stratified sample, separated features and target, and used a stratified train-test split.
4. **Transformation:** Standardized features inside the KNN pipeline using training data only.
5. **Data mining:** Trained baseline, Decision Tree, Random Forest, and KNN models.
6. **Evaluation:** Compared accuracy, macro precision, macro recall, macro F1-score, classification reports, and confusion matrix.
7. **Interpretation:** Examined class-level errors and Random Forest feature importance.

## 13. Conclusion (complete after running)

After executing all cells, use the printed metrics and plots to write your conclusion. State which model performed best on the held-out test set, whether it improved on the baseline, which classes were difficult to distinguish, and which features ranked highly. Clearly mention that the default results use a stratified sample of 80,000 records.

## References

1. Blackard, J. A., & Dean, D. J. (1999). Comparative accuracies of artificial neural networks and discriminant analysis in predicting forest cover types from cartographic variables. *Computers and Electronics in Agriculture, 24*(3), 131–151.
2. UCI Machine Learning Repository. *Covertype Dataset*. https://archive.ics.uci.edu/dataset/31/covertype
3. Scikit-learn documentation. *fetch_covtype*. https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html